# Synova Knowledge Base Creation

## Objective

Build and validate the multilingual knowledge base from structured JSON documents.

The workflow includes:

1. Imports.
2. Project Root.
3. Read all JSON files.
4. Convert JSON files to Chunks.
5. Create DataFrame.
6. Validation
7. Save knowledge_base.csv

# Import Libraries

In [9]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [10]:
KNOWLEDGE_BASE_DIR = PROJECT_ROOT / "knowledge_base"
PROCESSED_DIR = PROJECT_ROOT / "processed"

In [11]:
# جلب جميع ملفات JSON

json_files = list(
    KNOWLEDGE_BASE_DIR.rglob("*.json")
)

print(f"Found {len(json_files)} files.")

Found 24 files.


In [12]:
documents = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)

        documents.append(data)

print(len(documents))

24


In [13]:
# تحويل الـ JSON إلى Chunks
rows = []

for doc in documents:

    for section in doc["sections"]:

        content = section["content"]

        if isinstance(content, list):

            content = "\n".join(
                map(str, content)
            )

        rows.append({

            "id": doc["id"],

            "title": doc["title"],

            "language": doc["language"],

            "category": doc["category"],

            "section": section["title"],

            "text": content

        })

In [14]:
# إنشاء DataFrame
df = pd.DataFrame(rows)

df.head()

,id,title,language,category,section,text
0,hr_attendance,سياسة الدوام والحضور,ar,hr,نظرة عامة,يجب على جميع الموظفين الالتزام بساعات العمل ال...
1,hr_attendance,سياسة الدوام والحضور,ar,hr,ساعات العمل,يجب الالتزام بموعد بداية الدوام.\nيتم تسجيل ال...
2,hr_attendance,سياسة الدوام والحضور,ar,hr,المشكلات الشائعة,التأخر عن الدوام\nنسيان تسجيل الحضور\nنسيان تس...
3,hr_attendance,سياسة الدوام والحضور,ar,hr,إجراءات الحضور,سجل حضورك عند بداية الدوام.\nسجل انصرافك قبل م...
4,hr_attendance,سياسة الدوام والحضور,ar,hr,أفضل الممارسات,الالتزام بمواعيد الدوام.\nاستخدام نظام الحضور ...


In [15]:
print(df.shape)

df.groupby(
    ["category", "language"]
).size()

(156, 6)


category   language
hr         ar          36
           en          36
technical  ar          42
           en          42
dtype: int64

In [16]:
output_path = Path("../processed")

output_path.mkdir(
    exist_ok=True
)

df.to_csv(
    output_path / "knowledge_base.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully.")

Saved successfully.
